# Week 66, Day 5: concurrent.futures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from concurrent.futures import Future

f = Future()
print(f.done())
f.set_result(42)
print(f.done(), f.result())

g = Future()
g.set_exception(ValueError("bad input"))
print(g.done(), type(g.exception()).__name__)
try:
    g.result()
except ValueError as e:
    print("result() raised:", e)

In [ ]:
from concurrent.futures import Executor, Future

class InlineExecutor(Executor):
    "Runs each call immediately, in the calling thread."
    def submit(self, fn, /, *args, **kwargs):
        future = Future()
        try:
            future.set_result(fn(*args, **kwargs))
        except BaseException as e:
            future.set_exception(e)
        return future

with InlineExecutor() as pool:
    f = pool.submit(pow, 2, 10)
    print(f.result(), list(pool.map(str.upper, ["tea", "jam"])))
    print([pool.submit(len, w).result() for w in ("a", "bb", "ccc")])

In [ ]:
from concurrent.futures import Future, CancelledError

log = []
f = Future()
f.add_done_callback(lambda fut: log.append(("first", fut.result())))
f.set_result("ready")
f.add_done_callback(lambda fut: log.append(("late", fut.result())))   # runs at once
print(log)

pending = Future()
print(pending.cancel(), pending.cancelled(), pending.done())
try:
    pending.result()
except CancelledError:
    print("CancelledError")

slow = Future()
try:
    slow.result(timeout=0)
except TimeoutError:
    print("not finished yet")

In [ ]:
from concurrent.futures import Future, as_completed, wait, FIRST_COMPLETED

futures = {Future(): name for name in ("a", "b", "c")}
items = list(futures)
items[1].set_result("b finished first")
items[0].set_result("then a")
items[2].set_result("then c")
print(sorted(f.result() for f in as_completed(items)))

x, y = Future(), Future()
y.set_result(1)
done, not_done = wait([x, y], timeout=0, return_when=FIRST_COMPLETED)
print(len(done), len(not_done))
x.cancel()

In [ ]:
from concurrent.futures import Executor, Future

class InlineExecutor(Executor):
    def submit(self, fn, /, *args, **kwargs):
        f = Future()
        try:
            f.set_result(fn(*args, **kwargs))
        except BaseException as e:
            f.set_exception(e)
        return f

def parse(text):
    return int(text)

inputs = ["1", "2", "three", "4"]
with InlineExecutor() as pool:
    try:
        print(list(pool.map(parse, inputs)))
    except ValueError:
        print("map stopped at the first failure")
    futures = [pool.submit(parse, t) for t in inputs]
ok = [f.result() for f in futures if f.exception() is None]
failed = [t for t, f in zip(inputs, futures) if f.exception() is not None]
print(ok, failed)

In [ ]:
import pickle

def picklable(fn):
    try:
        pickle.dumps(fn)
        return True
    except Exception:
        return False

def square(x):
    return x * x

print(picklable(square), picklable(lambda x: x * x))

def choose(io_bound, task_ms, overhead_ms=5):
    if io_bound:
        return "ThreadPoolExecutor"
    return "ProcessPoolExecutor" if task_ms > 10 * overhead_ms else "run it in one process"

print(choose(True, 1), "|", choose(False, 500), "|", choose(False, 2))

> **Key idea.** Executors run callables on a pool of workers: submit returns a Future at once, map returns results in input order (stopping at the first exception), and with shuts the pool down after waiting for the work; ThreadPoolExecutor suits I/O-bound work and ProcessPoolExecutor CPU-bound work (picklable, importable functions, the __main__ guard, chunksize for small tasks). A Future holds a value or an exception: result(timeout) returns or re-raises, exception() returns it, cancel() works only before it starts, and add_done_callback runs a function on completion (at once if already done). as_completed yields futures as they finish and wait returns done and not-done sets. Handle errors per future, and remember a timeout stops waiting, not the task. Subclass Executor and provide submit to make your own.

### Exercise: Settling futures

Write `settle(future, func, *args)`: call `func(*args)` and store its result in the future, or store the exception if it raises, and return the future. Then `describe(future)`: `("pending", None)` if not done, `("cancelled", None)` if cancelled, `("error", "ValueError")` (the exception's type name) if it failed, or `("ok", value)`.

In [ ]:
from concurrent.futures import Future

def settle(future, func, *args):
    ___

def describe(future):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outcome():\n    \"futures\"\n    from concurrent.futures import Future\n    assert describe(Future()) == (\"pending\", None)\n    assert describe(settle(Future(), pow, 2, 5)) == (\"ok\", 32)\n    assert describe(settle(Future(), int, \"x\")) == (\"error\", \"ValueError\")\n    f = Future()\n    f.cancel()\n    assert describe(f) == (\"cancelled\", None)\n    g = settle(Future(), lambda: None)\n    assert describe(g) == (\"ok\", None) and g.done()")

### Exercise: Your own executor

Write `RecordingExecutor`, a subclass of `concurrent.futures.Executor` that runs each submitted call **immediately** and returns a settled `Future` (with the result or the exception). It records each call as `(function name, args)` in `calls`. After `shutdown()` (which the `with` statement calls), `submit` raises `RuntimeError("cannot schedule new futures after shutdown")`. The inherited `map` must work with it.

In [ ]:
from concurrent.futures import Executor, Future

class RecordingExecutor(Executor):
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_inline():\n    \"subclassing Executor\"\n    from concurrent.futures import as_completed\n    with RecordingExecutor() as ex:\n        f = ex.submit(pow, 3, 2)\n        assert f.done() and f.result() == 9\n        assert list(ex.map(len, [\"a\", \"bb\"])) == [1, 2]\n        bad = ex.submit(int, \"x\")\n        assert isinstance(bad.exception(), ValueError)\n        assert sorted(g.result() for g in as_completed([ex.submit(abs, -4), ex.submit(abs, 2)])) == [2, 4]\n    assert ex.calls[0] == (\"pow\", (3, 2)) and ex.calls[1] == (\"len\", (\"a\",)) and len(ex.calls) == 6\n    try:\n        ex.submit(print, 1)\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"cannot schedule new futures after shutdown\"")

### Exercise: Results and failures

Write `gather(executor, func, items)`: submit `func(item)` for every item, then return a tuple `(results, errors)`: `results` is a list in input order with `None` for failed items, and `errors` is a dict from the **index** of each failed item to its exception's type name. Use `submit` and each future's `exception()`, not `map`, so one failure doesn't hide the rest. The tests pass in an executor that runs calls at once.

In [ ]:
def gather(executor, func, items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gather():\n    \"per-task errors\"\n    from concurrent.futures import Executor, Future\n    class Inline(Executor):\n        def submit(self, fn, /, *args, **kwargs):\n            f = Future()\n            try:\n                f.set_result(fn(*args, **kwargs))\n            except BaseException as e:\n                f.set_exception(e)\n            return f\n    results, errors = gather(Inline(), int, [\"1\", \"x\", \"3\", None])\n    assert results == [1, None, 3, None] and errors == {1: \"ValueError\", 3: \"TypeError\"}\n    assert gather(Inline(), len, []) == ([], {})\n    assert \"map(\" not in source")

### Exercise: Input order or finishing order?

A pool with `workers` workers starts tasks in submission order, each on the worker free first (the lowest-numbered on ties). Given each task's duration, write `finish_times(durations, workers)` (a list, one per task), `as_completed_order(durations, workers)`: the task indices in the order `as_completed` would yield them (by finish time, ties by index), and `waiting_with_map(durations, workers)`: for each task, the time at which `map` would hand you its result, which is the latest finish time of that task and every task before it.

In [ ]:
def finish_times(durations, workers):
    ___

def as_completed_order(durations, workers):
    ___

def waiting_with_map(durations, workers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"as_completed versus map\"\n    d = [5, 1, 3, 1]\n    assert finish_times(d, 2) == [5, 1, 4, 5] and as_completed_order(d, 2) == [1, 2, 0, 3]\n    assert waiting_with_map(d, 2) == [5, 5, 5, 5]\n    assert as_completed_order([3, 2, 1], 3) == [2, 1, 0] and waiting_with_map([3, 2, 1], 3) == [3, 3, 3]\n    assert finish_times([2, 2, 2], 1) == [2, 4, 6] and as_completed_order([], 2) == []")

### Exercise: Chaining futures

Write `then(future, func)`: return a **new** future that, when `future` finishes, gets `func(future.result())`, or the same exception if `future` failed, or the exception `func` raises, or is cancelled if `future` was cancelled. Use `add_done_callback`, so it works whether `future` is already done or finishes later. Then `all_of(futures)`: a new future whose result is the list of all their results, settled when the last one finishes (with the first exception, in list order, if any failed).

In [ ]:
from concurrent.futures import Future

def then(future, func):
    ___

def all_of(futures):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chain():\n    \"callbacks\"\n    from concurrent.futures import Future\n    a = Future()\n    b = then(a, lambda x: x * 10)\n    c = then(b, str)\n    assert not c.done()\n    a.set_result(4)\n    assert b.result() == 40 and c.result() == \"40\"\n    done = Future()\n    done.set_result(1)\n    assert then(done, lambda x: x + 1).result() == 2\n    bad = Future()\n    bad.set_exception(KeyError(\"k\"))\n    assert isinstance(then(bad, str).exception(), KeyError)\n    assert isinstance(then(done, lambda x: 1 / 0).exception(), ZeroDivisionError)\n    gone = Future()\n    gone.cancel()\n    assert then(gone, str).cancelled()\n    fs = [Future(), Future(), Future()]\n    total = all_of(fs)\n    fs[2].set_result(\"c\"); fs[0].set_result(\"a\")\n    assert not total.done()\n    fs[1].set_result(\"b\")\n    assert total.result() == [\"a\", \"b\", \"c\"]\n    gs = [Future(), Future()]\n    failing = all_of(gs)\n    gs[1].set_exception(ValueError())\n    gs[0].set_result(1)\n    assert isinstance(failing.exception(), ValueError) and all_of([]).result() == []")

### Bug hunt: One failure loses the batch

`map` re-raises the first exception when the loop reaches it, so the results after it are never seen and this prints only `[1]` before the error is caught. Submit each item separately and keep the ones that succeed, so this prints `[1, 3]`.

In [ ]:
from concurrent.futures import Executor, Future

class Inline(Executor):
    def submit(self, fn, /, *args, **kwargs):
        f = Future()
        try:
            f.set_result(fn(*args, **kwargs))
        except BaseException as e:
            f.set_exception(e)
        return f

good = []
with Inline() as pool:
    try:
        for value in pool.map(int, ["1", "two", "3"]):
            good.append(value)
    except ValueError:
        pass
print(good)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"per-future handling\"\n    expect_output(\"[1, 3]\")")

### Bug hunt: Futures are not results

`as_completed` yields **futures**, not their values, so adding them to a number fails with `TypeError`. Call `result()` on each future so this prints `6`.

In [ ]:
from concurrent.futures import Future, as_completed

futures = []
for n in (1, 2, 3):
    f = Future()
    f.set_result(n)
    futures.append(f)

total = 0
for f in as_completed(futures):
    total += f
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"result()\"\n    expect_output(\"6\")")

### Bug hunt: Called too early

`pool.submit(square(4))` calls `square` **immediately**, in the current thread, and submits its result (the number 16) as if it were a function, so the future fails with `TypeError: 'int' object is not callable`. Pass the function and its argument separately so this prints `16`.

In [ ]:
from concurrent.futures import Executor, Future

class Inline(Executor):
    def submit(self, fn, /, *args, **kwargs):
        f = Future()
        try:
            f.set_result(fn(*args, **kwargs))
        except BaseException as e:
            f.set_exception(e)
        return f

def square(x):
    return x * x

with Inline() as pool:
    future = pool.submit(square(4))
print(future.result())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"submit(fn, arg)\"\n    expect_output(\"16\")")

### Puzzle: Handling results as they finish

Arrange the lines that report each download as soon as it finishes (as you'd write it on your own computer). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what executor.submit returns: "the result" or "a Future"
q = ___   # the order map gives results in: "input order" or "finishing order"
r = ___   # True or False: future.cancel() can stop a task that's already running
s = ___   # what future.result() does if the task raised: "returns None" or "re-raises"
t = ___   # the executor for CPU-bound pure-Python work: "ThreadPoolExecutor" or "ProcessPoolExecutor"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"a Future\" and q == \"input order\" and r is False, \"p, q and r\"\n    assert s == \"re-raises\" and t == \"ProcessPoolExecutor\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an executor with simulated time

Write `SimulatedPool(workers)`, an `Executor` subclass whose tasks take **simulated** time. `submit(fn, *args, duration=1, **kwargs)` returns a pending `Future` and queues the task; nothing runs until `run()`, which processes tasks like a real pool: each starts, in submission order, on the worker that becomes free first (lowest-numbered on ties); its function is called when it **starts** (so `fn` sees the right `now`), but its future is settled only at its finish time. Futures are settled in finish-time order (ties: submission order), so callbacks fire in that order. Also:

- `now` is the current simulated time (0 before running, the last finish time after);
- a future cancelled before `run()` is skipped and takes no time;
- `run()` returns a list of `(finish_time, future)` in the order they were settled;
- after `shutdown()`, `submit` raises `RuntimeError`; `shutdown()` (and so the `with` statement) calls `run()` if there's work left.

In [ ]:
import heapq
from concurrent.futures import Executor, Future

class SimulatedPool(Executor):
    def __init__(self, workers):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_simulated():\n    \"simulated time\"\n    from concurrent.futures import as_completed\n    pool = SimulatedPool(2)\n    order = []\n    fa = pool.submit(lambda: \"a\", duration=5)\n    fb = pool.submit(lambda: \"b\", duration=1)\n    fc = pool.submit(lambda: \"c\", duration=3)\n    for f, name in ((fa, \"a\"), (fb, \"b\"), (fc, \"c\")):\n        f.add_done_callback(lambda fut, name=name: order.append((name, pool.now)))\n    assert not fa.done() and pool.now == 0\n    settled = pool.run()\n    assert order == [(\"b\", 1), (\"c\", 4), (\"a\", 5)] and [t for t, _ in settled] == [1, 4, 5]\n    assert sorted(f.result() for f in as_completed([fa, fb, fc])) == [\"a\", \"b\", \"c\"] and pool.now == 5\n\ndef test_details():\n    \"starting times, errors, cancelling and shutdown\"\n    pool = SimulatedPool(1)\n    seen = []\n    def record(label):\n        seen.append((label, pool.now))\n        return label\n    f1 = pool.submit(record, \"first\", duration=2)\n    f2 = pool.submit(record, \"second\", duration=2)\n    f3 = pool.submit(int, \"x\", duration=1)\n    skipped = pool.submit(record, \"skipped\", duration=100)\n    assert skipped.cancel()\n    pool.run()\n    assert seen == [(\"first\", 0), (\"second\", 2)] and isinstance(f3.exception(), ValueError) and pool.now == 5\n    with SimulatedPool(3) as p:\n        g = p.submit(pow, 2, 8, duration=4)\n    assert g.result() == 256 and p.now == 4\n    try:\n        p.submit(abs, 1)\n        assert False\n    except RuntimeError:\n        pass")

## Warm-up

A future is a box for a result that may not exist yet. Once it's settled it holds either a value or an exception, and `result()` returns the value or **re-raises** the exception, in whichever thread asks. That's how errors travel back from workers.

## Executors

An **executor** owns a pool of workers and runs callables on them:

```
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=8) as pool:
    future = pool.submit(fetch, "https://example.org")   # returns at once
    page = future.result()                               # waits for this one
    pages = list(pool.map(fetch, urls))                  # results in input order
```

- `submit(fn, *args, **kwargs)` schedules `fn(*args, **kwargs)` and returns a `Future` at once.
- `map(fn, *iterables, timeout=None)` is like the built-in `map`: it returns an iterator of results **in input order** (waiting for each in turn), and re-raises the first exception it meets when you reach that item.
- The `with` block calls `shutdown(wait=True)` at the end, waiting for all submitted work; after shutdown, `submit` raises `RuntimeError`.
- `ThreadPoolExecutor` uses threads (good for I/O-bound work); `ProcessPoolExecutor` uses processes (CPU-bound work: arguments and results are pickled, Monday, and the call must be importable, so not a lambda). Both default to a sensible number of workers.

`Executor` is an abstract base class (Week 32): a subclass must provide `submit`, and inherits `map` and the context manager. Writing one that runs calls at once is a good way to test code that uses executors, and it works here:

## Futures

A `Future` has a small, precise interface:

- **State**: `done()` (finished, failed or cancelled), `running()`, `cancelled()`.
- **Outcome**: `result(timeout=None)` waits up to `timeout` seconds and returns the value, re-raises the task's exception, raises `CancelledError` if cancelled, or raises `TimeoutError` if time runs out; `exception(timeout=None)` returns the exception (or `None`) instead of raising it.
- **Callbacks**: `add_done_callback(fn)` calls `fn(future)` when it finishes, or at once if it already has. Callbacks run in the thread that completed the future, so keep them short and don't raise from them.
- **Settling** (for executors and tests, not normal user code): `set_result(value)`, `set_exception(exc)`, and `set_running_or_notify_cancel()`.

Futures are a general idea, not just a Python one: JavaScript calls them promises, and next week's `asyncio` has its own futures with the same shape.

## As they complete

`map` gives results in **input** order, so one slow task holds up all the results after it. Often you'd rather handle each result **as soon as it's ready**:

- `as_completed(futures, timeout=None)` yields futures **in the order they finish**. Keep a dict from future to its input so you know which is which.
- `wait(futures, timeout=None, return_when=ALL_COMPLETED)` returns two sets, `(done, not_done)`; `return_when` can be `FIRST_COMPLETED` or `FIRST_EXCEPTION`.

The usual pattern for a batch of I/O work, on your own computer:

```
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(fetch, url): url for url in urls}
    for future in as_completed(futures):
        url = futures[future]
        try:
            page = future.result()
        except Exception as e:
            print(url, "failed:", e)
        else:
            print(url, len(page), "bytes")
```

## Errors, timeouts and cancelling

- **One failure shouldn't sink a batch.** With `map`, the first exception stops the iteration and the remaining results are lost to you. With `submit` and `as_completed`, check each future: `future.exception()` or `try: future.result()`.
- **Timeouts**: `future.result(timeout=5)` gives up waiting after five seconds, but the task **keeps running** in its worker: Python can't stop a running thread. Design tasks to time themselves out (network calls usually accept a timeout).
- **Cancelling**: `future.cancel()` succeeds only if the task hasn't started; a running or finished task can't be cancelled. `executor.shutdown(cancel_futures=True)` (3.9+) cancels everything still queued.
- **Exceptions inside a process pool** are pickled and re-raised in the parent; the original traceback is attached as text.

## Threads or processes?

The decision follows Monday's rules:

| | ThreadPoolExecutor | ProcessPoolExecutor |
|---|---|---|
| good for | I/O-bound work: network, files, databases | CPU-bound pure-Python work |
| parallel Python code? | no (with the GIL) | yes: one interpreter each |
| start-up cost | low | higher (new processes) |
| arguments and results | shared, no copying | pickled: must be picklable, copied |
| functions | anything callable | must be importable (top-level functions, not lambdas) |
| where it fails | CPU-bound work gains nothing | tiny tasks: overheads dominate |

Two practical notes for process pools: guard the code that creates them with `if __name__ == "__main__":`, and send work in **chunks** (`map(fn, items, chunksize=100)`) when tasks are small, so each round trip carries enough work.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, rewrite Monday's downloader with `ThreadPoolExecutor` and `as_completed`, and make one of the fake downloads raise an exception: check that the others still report. Tomorrow is the Forge: a deterministic web-crawler simulation using everything from this week.